# Brackish: enhancement export and matched YOLOv8 comparison

This notebook builds one correctly labelled Brackish detection dataset from the official COCO annotations and videos, exports baseline and ERG enhanced copies, then trains and evaluates three matched YOLOv8 experiments: **Original**, **UICE-MIRNet**, and **ERG-UICE-MIRNet**.

> **Draft safety:** every action that creates images, starts enhancement, or starts YOLO training is disabled by default. Review the paths and change the relevant flag to `True` before manually running that section. This notebook has not been run by Codex.

The existing `annotations_YOLO` text files are not used: their coordinates are in pixels, while YOLO requires normalized coordinates. Labels are generated correctly from the official COCO bounding boxes instead.

## 1. Imports and configuration

Use the CUDA-enabled project environment. If `ultralytics` or `cv2` is missing there, install them manually in that environment before enabling the later sections:

```python
%pip install ultralytics opencv-python
```

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import os
import shutil
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# This works when the notebook is opened from files (6) or the workspace root.
cwd = Path.cwd().resolve()
if (cwd / 'model.py').is_file():
    PROJECT_DIR = cwd
elif (cwd / 'UICE-CV' / 'files (6)' / 'model.py').is_file():
    PROJECT_DIR = cwd / 'UICE-CV' / 'files (6)'
else:
    raise RuntimeError('Run from UICE-CV/files (6) or the workspace root.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

REPO_DIR = PROJECT_DIR.parent
BRACKISH_DIR = REPO_DIR / 'brackish'
COCO_DIR = BRACKISH_DIR / 'annotations' / 'annotations_COCO'
VIDEO_DIR = BRACKISH_DIR / 'dataset' / 'videos'
RUN_TAG = 'brackish_restart_100epoch_seed42_fp32'
CHECKPOINT_ROOT = REPO_DIR / 'checkpoints' / RUN_TAG
BASELINE_CHECKPOINT = CHECKPOINT_ROOT / 'baseline' / 'uice_mirnet_epoch20.pth'
ERG_CHECKPOINT = CHECKPOINT_ROOT / 'erg' / 'uice_mirnet_epoch20.pth'

# All generated YOLO data, YAML files, model runs and final CSV live here.
OUTPUT_ROOT = REPO_DIR / 'results' / 'brackish_yolov8_epoch20'
DATASETS_ROOT = OUTPUT_ROOT / 'datasets'
RUNS_ROOT = OUTPUT_ROOT / 'runs'

# The annotation names use a 1-based frame number (for example ...-0001.png).
# OpenCV uses zero-based positions, hence -1. Keep this unchanged unless a
# visual spot check proves that the annotations are offset.
ANNOTATION_FRAME_OFFSET = -1
CLASS_NAMES = ['fish', 'small_fish', 'crab', 'shrimp', 'jellyfish', 'starfish']
SPLIT_TO_COCO = {
    'train': COCO_DIR / 'train_groundtruth.json',
    'val': COCO_DIR / 'valid_groundtruth.json',
    'test': COCO_DIR / 'test_groundtruth.json',
}

# Leave all actions off while reviewing this draft. Enable in order: build,
# export, then train. The notebook never deletes existing data automatically.
BUILD_ORIGINAL_DATASET = False
EXPORT_ENHANCED_DATASETS = False
RUN_YOLO_TRAINING = False

# Enhancement export options. One full-resolution image at a time avoids GPU OOM.
ENHANCEMENT_DEVICE = 'cuda'

# Matched YOLOv8 settings. All three experiments must retain these values.
YOLO_WEIGHTS = 'yolov8n.pt'  # Change once here only if using another YOLOv8 size.
YOLO_EPOCHS = 100
YOLO_IMAGE_SIZE = 640
YOLO_BATCH = -1             # Ultralytics automatic GPU batch-size selection.
YOLO_WORKERS = min(8, max(1, os.cpu_count() or 1))
YOLO_DEVICE = 0
YOLO_SEED = 42

print('Output root:', OUTPUT_ROOT)
print('Actions enabled:', {
    'build_original': BUILD_ORIGINAL_DATASET,
    'export_enhanced': EXPORT_ENHANCED_DATASETS,
    'train_yolo': RUN_YOLO_TRAINING,
})

Output root: C:\PYTHON 2\UICE-CV\results\brackish_yolov8_epoch20
Actions enabled: {'build_original': False, 'export_enhanced': False, 'train_yolo': False}


## 2. Read and validate the official annotations

Brackish's supplied COCO train/validation/test protocol is **frame-level**, not sequence-disjoint: different frames from the same videos occur in every split. The code therefore verifies that no exact image is duplicated across splits, reports the video overlap as an explicit warning, and ensures every image name resolves to exactly one source video before writing dataset files.

In [ ]:
def stem_and_frame_number(file_name):
    stem = Path(file_name).stem
    video_stem, separator, frame_number = stem.rpartition('-')
    if not separator or not frame_number.isdigit():
        raise ValueError(f'Unexpected Brackish annotation name: {file_name}')
    return video_stem, int(frame_number)

video_paths = list(VIDEO_DIR.rglob('*.avi'))
VIDEO_BY_STEM = {}
for video_path in video_paths:
    if video_path.stem in VIDEO_BY_STEM:
        raise RuntimeError(f'Duplicate video stem: {video_path.stem}')
    VIDEO_BY_STEM[video_path.stem] = video_path

records_by_split = {}
category_names = None
for split, coco_path in SPLIT_TO_COCO.items():
    if not coco_path.is_file():
        raise FileNotFoundError(f'Missing COCO annotations: {coco_path}')
    with coco_path.open(encoding='utf-8') as handle:
        coco = json.load(handle)
    current_categories = {item['id']: item['name'] for item in coco['categories']}
    if category_names is None:
        category_names = current_categories
    elif category_names != current_categories:
        raise RuntimeError('COCO category mapping differs between splits.')
    annotations_by_image = defaultdict(list)
    for annotation in coco['annotations']:
        annotations_by_image[annotation['image_id']].append(annotation)
    records = []
    for image in coco['images']:
        video_stem, frame_number = stem_and_frame_number(image['file_name'])
        if video_stem not in VIDEO_BY_STEM:
            raise FileNotFoundError(f'No source video found for {image["file_name"]}')
        records.append({
            'file_name': image['file_name'],
            'image_id': image['id'],
            'width': image['width'],
            'height': image['height'],
            'video_stem': video_stem,
            'frame_number': frame_number,
            'annotations': annotations_by_image[image['id']],
        })
    records_by_split[split] = sorted(records, key=lambda item: item['file_name'])

expected_names = {index: name for index, name in enumerate(CLASS_NAMES)}
coco_names = {category_id - 1: name for category_id, name in category_names.items()}
if coco_names != expected_names:
    raise RuntimeError(f'Unexpected class mapping: {coco_names}')

sequences = {split: {record['video_stem'] for record in records} for split, records in records_by_split.items()}
file_names = {split: {record['file_name'] for record in records} for split, records in records_by_split.items()}
for first, second in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    duplicate_images = file_names[first] & file_names[second]
    if duplicate_images:
        raise RuntimeError(
            f'Exact image leakage between {first} and {second}: {sorted(duplicate_images)[:5]}'
        )
    shared_videos = sequences[first] & sequences[second]
    if shared_videos:
        print(
            f'WARNING: Official Brackish frame-level split shares {len(shared_videos)} video sequences '
            f'between {first} and {second}. No exact images are duplicated.'
        )

for split, records in records_by_split.items():
    print(f'{split}: {len(records)} images, {sum(len(item["annotations"]) for item in records)} boxes, {len(sequences[split])} videos')

## 3. Build the original, correctly labelled YOLO dataset

This exports the annotated original frames from the videos and writes zero-based, normalized `class x_center y_center width height` labels. The source videos are 1920x1080 while official annotations are 960x540, so each frame is resized to the annotation resolution before it is saved; this preserves box alignment. Empty-label images are preserved as negative examples. Existing files are not overwritten; delete or choose a new output directory yourself if you intentionally need a clean rebuild.

In [ ]:
def yolo_label_lines(record, frame_width, frame_height):
    lines = []
    for annotation in record['annotations']:
        category_id = int(annotation['category_id']) - 1
        if category_id not in range(len(CLASS_NAMES)):
            raise ValueError(f'Invalid COCO class id: {annotation["category_id"]}')
        x, y, width, height = map(float, annotation['bbox'])
        x1, y1 = max(0.0, x), max(0.0, y)
        x2, y2 = min(float(frame_width), x + width), min(float(frame_height), y + height)
        if x2 <= x1 or y2 <= y1:
            raise ValueError(f'Invalid/clipped box in {record["file_name"]}: {annotation["bbox"]}')
        x_center = ((x1 + x2) / 2.0) / frame_width
        y_center = ((y1 + y2) / 2.0) / frame_height
        normalized_width = (x2 - x1) / frame_width
        normalized_height = (y2 - y1) / frame_height
        values = (x_center, y_center, normalized_width, normalized_height)
        if not all(0.0 < value <= 1.0 for value in values):
            raise ValueError(f'Out-of-range YOLO box in {record["file_name"]}: {values}')
        lines.append(f'{category_id} {x_center:.8f} {y_center:.8f} {normalized_width:.8f} {normalized_height:.8f}')
    return lines

def original_split_dirs(split):
    root = DATASETS_ROOT / 'original'
    return root / 'images' / split, root / 'labels' / split

def build_original_split(split, records):
    image_dir, label_dir = original_split_dirs(split)
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    records_by_video = defaultdict(list)
    for record in records:
        records_by_video[record['video_stem']].append(record)
    written = 0
    for video_stem, video_records in sorted(records_by_video.items()):
        capture = cv2.VideoCapture(str(VIDEO_BY_STEM[video_stem]))
        if not capture.isOpened():
            raise RuntimeError(f'Could not open video: {VIDEO_BY_STEM[video_stem]}')
        try:
            for record in video_records:
                stem = Path(record['file_name']).stem
                image_path = image_dir / f'{stem}.jpg'
                label_path = label_dir / f'{stem}.txt'
                if image_path.exists() or label_path.exists():
                    raise FileExistsError(
                        f'Refusing to overwrite existing export: {image_path}. '
                        'Use a new OUTPUT_ROOT or manually remove only the intended export.'
                    )
                frame_position = record['frame_number'] + ANNOTATION_FRAME_OFFSET
                if frame_position < 0:
                    raise ValueError(f'Negative frame position for {record["file_name"]}')
                capture.set(cv2.CAP_PROP_POS_FRAMES, frame_position)
                ok, frame = capture.read()
                if not ok:
                    raise RuntimeError(f'Could not read frame {frame_position} from {VIDEO_BY_STEM[video_stem]}')
                frame_height, frame_width = frame.shape[:2]
                annotation_width, annotation_height = int(record['width']), int(record['height'])
                video_aspect = frame_width / frame_height
                annotation_aspect = annotation_width / annotation_height
                if not np.isclose(video_aspect, annotation_aspect, rtol=1e-3):
                    raise RuntimeError(
                        f'Aspect-ratio mismatch for {record["file_name"]}: '
                        f'video={frame_width}x{frame_height}, annotation={annotation_width}x{annotation_height}'
                    )
                if (frame_width, frame_height) != (annotation_width, annotation_height):
                    frame = cv2.resize(frame, (annotation_width, annotation_height), interpolation=cv2.INTER_AREA)
                frame_height, frame_width = frame.shape[:2]
                if not cv2.imwrite(str(image_path), frame):
                    raise RuntimeError(f'Could not write {image_path}')
                label_path.write_text('\n'.join(yolo_label_lines(record, frame_width, frame_height)), encoding='utf-8')
                written += 1
        finally:
            capture.release()
    return written

if BUILD_ORIGINAL_DATASET:
    counts = {split: build_original_split(split, records) for split, records in records_by_split.items()}
    print('Original YOLO export complete:', counts)
else:
    print('Not run: set BUILD_ORIGINAL_DATASET = True after reviewing the configuration.')

## 3b. Mandatory visual annotation spot check

Run this immediately after building the original dataset and **before** exporting enhanced images. The boxes must align with the objects. If they do not, stop and correct `ANNOTATION_FRAME_OFFSET`; do not continue to YOLO training.

In [ ]:
def draw_yolo_annotations(image_path, label_path):
    with Image.open(image_path) as image:
        canvas = np.asarray(image.convert('RGB')).copy()
    height, width = canvas.shape[:2]
    for line in label_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        class_id, x_center, y_center, box_width, box_height = map(float, line.split())
        x1 = int((x_center - box_width / 2) * width)
        y1 = int((y_center - box_height / 2) * height)
        x2 = int((x_center + box_width / 2) * width)
        y2 = int((y_center + box_height / 2) * height)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (255, 80, 0), 2)
        cv2.putText(canvas, CLASS_NAMES[int(class_id)], (x1, max(20, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 80, 0), 2, cv2.LINE_AA)
    return canvas

test_image_dir, test_label_dir = original_split_dirs('test')
candidates = [
    path for path in sorted(test_image_dir.glob('*.jpg'))
    if (test_label_dir / f'{path.stem}.txt').stat().st_size > 0
]
if not candidates:
    raise RuntimeError('No labelled test images found. Build the original dataset first.')
spot_check_path = candidates[0]
annotated = draw_yolo_annotations(spot_check_path, test_label_dir / f'{spot_check_path.stem}.txt')
plt.figure(figsize=(14, 8))
plt.imshow(annotated)
plt.title(f'Annotation spot check: {spot_check_path.name}')
plt.axis('off')
plt.show()

## 4. Export matching baseline and ERG enhanced datasets

The original labels are copied unchanged. Images retain their filename and dimensions, so each enhanced image has exactly the same bounding boxes as its original. The code loads the saved, separate epoch-20 checkpoints and will reject a baseline checkpoint mistakenly used as ERG.

In [ ]:
if EXPORT_ENHANCED_DATASETS:
    import torch
    from model import UICEMIRNet
    from erg_uice_mirnet import ERGUICEMIRNet

    if ENHANCEMENT_DEVICE == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA was requested but is unavailable in this notebook kernel.')
    device = torch.device(ENHANCEMENT_DEVICE if torch.cuda.is_available() else 'cpu')

    def load_checkpoint(path, expects_erg):
        if not path.is_file():
            raise FileNotFoundError(f'Missing enhancement checkpoint: {path}')
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
        if not isinstance(checkpoint, dict) or 'model_state' not in checkpoint:
            raise RuntimeError(f'Unsupported checkpoint format: {path}')
        is_erg = 'erg_fusion_mode' in checkpoint.get('args', {})
        if is_erg != expects_erg:
            expected = 'ERG-UICE-MIRNet' if expects_erg else 'baseline UICE-MIRNet'
            raise RuntimeError(f'{path.name} is not a {expected} checkpoint.')
        return checkpoint

    def architecture(checkpoint):
        args = checkpoint.get('args', {})
        return int(args.get('channels', 32)), int(args.get('num_rrb', 3))

    baseline_checkpoint = load_checkpoint(BASELINE_CHECKPOINT, expects_erg=False)
    erg_checkpoint = load_checkpoint(ERG_CHECKPOINT, expects_erg=True)
    baseline_channels, baseline_rrbs = architecture(baseline_checkpoint)
    erg_channels, erg_rrbs = architecture(erg_checkpoint)
    erg_args = erg_checkpoint.get('args', {})

    baseline_model = UICEMIRNet(channels=baseline_channels, num_rrb=baseline_rrbs)
    baseline_model.load_state_dict(baseline_checkpoint['model_state'], strict=True)
    erg_model = ERGUICEMIRNet(
        channels=erg_channels,
        num_rrb=erg_rrbs,
        reliability_channels=int(erg_args.get('erg_reliability_channels', 16)),
        fusion_mode=erg_args.get('erg_fusion_mode', 'feature'),
    )
    erg_model.load_state_dict(erg_checkpoint['model_state'], strict=True)
    baseline_model = baseline_model.to(device).eval()
    erg_model = erg_model.to(device).eval()

    @torch.inference_mode()
    def enhance_to_file(model, source_path, destination_path):
        with Image.open(source_path) as image:
            rgb = image.convert('RGB')
            image_array = np.asarray(rgb, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(image_array).permute(2, 0, 1).unsqueeze(0).to(device)
        output = model(tensor).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
        output_array = (output * 255.0).round().astype(np.uint8)
        Image.fromarray(output_array, mode='RGB').save(destination_path, quality=95)

    def export_variant(variant_name, model):
        for split in SPLIT_TO_COCO:
            source_images, source_labels = original_split_dirs(split)
            if not source_images.is_dir() or not source_labels.is_dir():
                raise FileNotFoundError('Build the original dataset before exporting enhancements.')
            target_images = DATASETS_ROOT / variant_name / 'images' / split
            target_labels = DATASETS_ROOT / variant_name / 'labels' / split
            target_images.mkdir(parents=True, exist_ok=True)
            target_labels.mkdir(parents=True, exist_ok=True)
            for source_path in sorted(source_images.glob('*.jpg')):
                output_path = target_images / source_path.name
                label_source = source_labels / f'{source_path.stem}.txt'
                label_destination = target_labels / label_source.name
                if output_path.exists() or label_destination.exists():
                    raise FileExistsError(f'Refusing to overwrite export: {output_path}')
                enhance_to_file(model, source_path, output_path)
                shutil.copy2(label_source, label_destination)
            print(f'{variant_name}/{split}: {len(list(target_images.glob("*.jpg")))} images exported')

    export_variant('uice_mirnet', baseline_model)
    export_variant('erg_uice_mirnet', erg_model)
    print('Enhanced dataset export complete.')
else:
    print('Not run: set EXPORT_ENHANCED_DATASETS = True after the original dataset is complete.')

## 5. Verify matched files and write the three YOLO dataset YAML files

Run this after export. It confirms that every image has one label file and that the three variants contain exactly the same filenames before training begins.

In [ ]:
VARIANTS = {
    'original': DATASETS_ROOT / 'original',
    'uice_mirnet': DATASETS_ROOT / 'uice_mirnet',
    'erg_uice_mirnet': DATASETS_ROOT / 'erg_uice_mirnet',
}

def split_stems(dataset_root, split):
    image_dir = dataset_root / 'images' / split
    label_dir = dataset_root / 'labels' / split
    image_stems = {path.stem for path in image_dir.glob('*.jpg')}
    label_stems = {path.stem for path in label_dir.glob('*.txt')}
    if image_stems != label_stems:
        raise RuntimeError(
            f'Image/label mismatch for {dataset_root.name}/{split}: '
            f'images only={sorted(image_stems - label_stems)[:3]}, '
            f'labels only={sorted(label_stems - image_stems)[:3]}'
        )
    return image_stems

all_stems = {}
for variant, dataset_root in VARIANTS.items():
    all_stems[variant] = {}
    for split in SPLIT_TO_COCO:
        all_stems[variant][split] = split_stems(dataset_root, split)
        print(f'{variant}/{split}: {len(all_stems[variant][split])} matched image-label pairs')

for split in SPLIT_TO_COCO:
    reference = all_stems['original'][split]
    for variant in ('uice_mirnet', 'erg_uice_mirnet'):
        if all_stems[variant][split] != reference:
            raise RuntimeError(f'Variant filename mismatch: {variant}/{split}')

def write_dataset_yaml(variant, dataset_root):
    yaml_path = OUTPUT_ROOT / f'brackish_{variant}.yaml'
    lines = [
        f'path: {dataset_root.as_posix()}',
        'train: images/train',
        'val: images/val',
        'test: images/test',
        'names:',
    ] + [f'  {index}: {name}' for index, name in enumerate(CLASS_NAMES)]
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    yaml_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return yaml_path

DATA_YAMLS = {variant: write_dataset_yaml(variant, root) for variant, root in VARIANTS.items()}
DATA_YAMLS

## 6. Train and evaluate the three matched YOLOv8 models

Each model is initialized from the same YOLOv8 weights and uses the same hyperparameters and seed. During training, Ultralytics selects the best epoch by validation performance. The held-out test split is used only once per model for the final reported metrics.

In [ ]:
if RUN_YOLO_TRAINING:
    from ultralytics import YOLO

    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    experiments = [
        ('original', 'Original'),
        ('uice_mirnet', 'UICE-MIRNet'),
        ('erg_uice_mirnet', 'ERG-UICE-MIRNet'),
    ]
    metric_rows = []
    for variant, display_name in experiments:
        run_name = f'brackish_{variant}'
        run_dir = RUNS_ROOT / run_name
        if run_dir.exists():
            raise FileExistsError(
                f'Run directory already exists: {run_dir}. '
                'Choose a new OUTPUT_ROOT or rename the run; this notebook will not overwrite runs.'
            )
        model = YOLO(YOLO_WEIGHTS)
        model.train(
            data=str(DATA_YAMLS[variant]),
            epochs=YOLO_EPOCHS,
            imgsz=YOLO_IMAGE_SIZE,
            batch=YOLO_BATCH,
            workers=YOLO_WORKERS,
            device=YOLO_DEVICE,
            seed=YOLO_SEED,
            deterministic=True,
            project=str(RUNS_ROOT),
            name=run_name,
            exist_ok=False,
            pretrained=True,
            plots=True,
        )
        best_weights = run_dir / 'weights' / 'best.pt'
        if not best_weights.is_file():
            raise FileNotFoundError(f'YOLO training did not create best weights: {best_weights}')
        test_model = YOLO(str(best_weights))
        test_metrics = test_model.val(
            data=str(DATA_YAMLS[variant]),
            split='test',
            imgsz=YOLO_IMAGE_SIZE,
            batch=YOLO_BATCH,
            workers=YOLO_WORKERS,
            device=YOLO_DEVICE,
            project=str(RUNS_ROOT),
            name=f'{run_name}_test',
            exist_ok=False,
            plots=True,
        )
        metric_rows.append({
            'experiment': display_name,
            'variant': variant,
            'weights': str(best_weights),
            'precision': float(test_metrics.box.mp),
            'recall': float(test_metrics.box.mr),
            'mAP50': float(test_metrics.box.map50),
            'mAP50_95': float(test_metrics.box.map),
        })
    comparison = pd.DataFrame(metric_rows)
    comparison_path = OUTPUT_ROOT / 'yolov8_test_metric_comparison.csv'
    comparison.to_csv(comparison_path, index=False)
    display(comparison)
    print(f'Saved final comparison: {comparison_path}')
else:
    print('Not run: set RUN_YOLO_TRAINING = True only after all three datasets validate.')

## 7. View the final metric comparison

This cell is safe to run after the three training runs finish. It reads the saved test-metric CSV only.

In [ ]:
comparison_path = OUTPUT_ROOT / 'yolov8_test_metric_comparison.csv'
if comparison_path.is_file():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
    plot_data = comparison.set_index('experiment')[['precision', 'recall', 'mAP50', 'mAP50_95']]
    ax = plot_data.plot(kind='bar', figsize=(11, 5), ylim=(0, 1), rot=0)
    ax.set_ylabel('Score')
    ax.set_title('Brackish held-out test: matched YOLOv8 detection comparison')
    ax.legend(loc='lower right')
    plt.tight_layout()
    figure_path = OUTPUT_ROOT / 'yolov8_test_metric_comparison.png'
    plt.savefig(figure_path, dpi=250)
    plt.show()
    print(f'Saved comparison figure: {figure_path}')
else:
    print(f'No metrics file yet: {comparison_path}')